<a href="https://colab.research.google.com/github/Camacho-umu/CamachoMu-ozRuiz/blob/main/k_brazos/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parte 1: Problema del Bandido de k-Brazos

**Asignatura:** Extensiones de Machine Learning — Máster IA (UMU)

---

## Introducción

El problema del **bandido de k-brazos** (*k-armed bandit*) es un caso fundamental del aprendizaje por refuerzo en el que un agente debe elegir repetidamente entre $k$ acciones (brazos), cada una con una distribución de recompensa desconocida. El objetivo es maximizar la recompensa acumulada, lo que requiere equilibrar **exploración** (probar brazos para estimar sus valores) y **explotación** (elegir el brazo con mayor valor estimado).

### Algoritmos estudiados

| Algoritmo | Hiperparámetro | Estrategia |
|---|---|---|
| **ε-greedy** | ε (prob. exploración) | Con probabilidad ε explora aleatoriamente; con 1-ε explota el mejor brazo conocido |
| **UCB1** | c (factor exploración) | Selecciona el brazo que maximiza $Q(a) + c\sqrt{\frac{\ln t}{N(a)}}$ |
| **Softmax** | τ (temperatura) | Selecciona proporcionalmente a $\frac{e^{Q(a)/\tau}}{\sum_b e^{Q(b)/\tau}}$ |

### Distribuciones de brazos

Se estudian tres familias de distribuciones:
- **Normal** $N(\mu, \sigma)$: recompensas continuas en $(-\infty, +\infty)$.
- **Bernoulli** $\text{Ber}(p)$: recompensas binarias en $\{0, 1\}$.
- **Binomial** $\text{Bin}(n, p)$: recompensas enteras en $\{0, \ldots, n\}$, conectando Bernoulli y Normal.

In [ ]:
# Configuración para Google Colab
import os

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('CamachoMu-ozRuiz'):
        !git clone https://github.com/Camacho-umu/CamachoMu-ozRuiz.git
    %cd CamachoMu-ozRuiz/k_brazos
    !pip install -q -r ../requirements.txt

import sys
sys.path.insert(0, 'src')
print("Entorno configurado correctamente.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from arms import ArmNormal, ArmBernoulli, ArmBinomial, Bandit
from algorithms import EpsilonGreedy, UCB1, Softmax
from plotting import plot_average_rewards, plot_optimal_selections, plot_regret, plot_arm_statistics
from main import run_experiment

SEED = 42
print("Librerías importadas correctamente.")

---
## Demostración rápida: ε-greedy con brazos Normal

Ejecución base con 3 variantes de ε-greedy sobre un bandido de 10 brazos con distribución Normal, para verificar que el entorno funciona correctamente.

In [ ]:
np.random.seed(SEED)

k, steps, runs = 10, 1000, 500
bandit = Bandit(arms=ArmNormal.generate_arms(k))

print(f"Bandido de {k} brazos (Normal):")
for i, arm in enumerate(bandit.arms):
    marker = " ★ ÓPTIMO" if i == bandit.optimal_arm else ""
    print(f"  Brazo {i+1}: μ = {arm.mu:.3f}, σ = {arm.sigma:.3f}{marker}")

algorithms = [
    EpsilonGreedy(k=k, epsilon=0),
    EpsilonGreedy(k=k, epsilon=0.01),
    EpsilonGreedy(k=k, epsilon=0.1),
]

print(f"\nEjecutando {runs} ejecuciones × {steps} pasos...")
rew, opt, reg, arm_stats = run_experiment(bandit, algorithms, steps, runs)
print("Completado.")

In [ ]:
plot_average_rewards(steps, rew, algorithms)
plot_optimal_selections(steps, opt, algorithms)

---
## Estudios comparativos detallados

A continuación se enlazan los notebooks con los estudios comparativos completos de los tres algoritmos sobre cada distribución de brazos. En cada estudio se realiza un barrido de hiperparámetros y una comparativa final con los mejores valores.

| Notebook | Distribución | Descripción |
|---|---|---|
| [estudio_comparativo.ipynb](estudio_comparativo.ipynb) | Normal + Bernoulli + Binomial | Comparativa global: barrido de hiperparámetros y comparativa final para las 3 distribuciones |
| [estudio_comparativo_normal.ipynb](estudio_comparativo_normal.ipynb) | Normal $N(\mu, \sigma)$ | Estudio detallado con brazos Normal |
| [estudio_comparativo_bernoulli.ipynb](estudio_comparativo_bernoulli.ipynb) | Bernoulli $\text{Ber}(p)$ | Estudio detallado con brazos Bernoulli |
| [estudio_comparativo_binomial.ipynb](estudio_comparativo_binomial.ipynb) | Binomial $\text{Bin}(n,p)$ | Estudio detallado con brazos Binomial, efecto de n en la señal |